#  Sistema RAG: Preguntas y Respuestas sobre Documentos

> **Retrieval Augmented Generation aplicado a documentos en español e inglés.**

En esta sesión vamos a construir, paso a paso, un sistema que **lee un documento, lo entiende y responde preguntas** sobre su contenido — todo desde un Jupyter Notebook.

###  Stack tecnológico

| Capa | Herramienta |
|---|---|
| Extracción de PDF | `pypdf` |
| Tokenizador | `tiktoken` (cl100k_base) |
| Segmentación | `langchain-text-splitters` |
| Traducción | `deep-translator` (Google web) |
| Embeddings + LLM | `google-genai` (Gemini) |
| Base vectorial | `chromadb` |
| Interfaz | `ipywidgets` |

---

>  **Para esta demostración usaremos documentos cortos** (5–30 páginas) para que el flujo completo se ejecute en pocos minutos durante la clase.

##  Paso 0 — Preparación del entorno

Antes de empezar, verifica que tienes instaladas las librerías. Si trabajas con un entorno virtual recién creado, ejecuta la celda de instalación. Si ya las tienes, puedes saltarla.

In [1]:
# Descomentar la siguiente línea solo si necesitas instalar las dependencias
!pip install pypdf tiktoken langchain-text-splitters deep-translator google-genai chromadb ipywidgets tqdm python-dotenv

  Using cached pypdf-6.10.2-py3-none-any.whl.metadata (7.1 kB)
  Using cached langchain_text_splitters-1.1.2-py3-none-any.whl.metadata (3.3 kB)
  Using cached deep_translator-1.11.4-py3-none-any.whl.metadata (30 kB)
  Using cached google_genai-1.73.1-py3-none-any.whl.metadata (52 kB)
  Using cached chromadb-1.5.8-cp39-abi3-win_amd64.whl.metadata (5.1 kB)
  Using cached langchain_core-1.3.2-py3-none-any.whl.metadata (4.4 kB)
  Using cached langchain_protocol-0.0.13-py3-none-any.whl.metadata (2.4 kB)
  Using cached uuid_utils-0.14.1-cp39-abi3-win_amd64.whl.metadata (4.9 kB)
  Using cached orjson-3.11.8-cp313-cp313-win_amd64.whl.metadata (43 kB)
  Using cached google_auth-2.49.2-py3-none-any.whl.metadata (6.2 kB)
  Using cached build-1.4.4-py3-none-any.whl.metadata (5.8 kB)
  Using cached pybase64-1.4.3-cp313-cp313-win_amd64.whl.metadata (9.1 kB)
  Using cached uvicorn-0.46.0-py3-none-any.whl.metadata (6.7 kB)
  Using cached onnxruntime-1.25.1-cp313-cp313-win_amd64.whl.metadata (5.5 kB)
 

##  Paso 1 — Carga segura de la API Key de Gemini

Para usar los modelos de Google (embeddings y chat) necesitas una clave gratuita de **Google AI Studio**:  
 <https://aistudio.google.com/app/apikey>

>  El plan gratuito de `gemini-embedding-001` tiene un límite de **~60 peticiones por minuto**. Por eso, más adelante, agregaremos una pequeña pausa entre llamadas.

**Cómo se carga la clave (en orden de prioridad):**

1. Variable de entorno `GEMINI_API_KEY` ya definida en el sistema.
2. Archivo `.env` en la carpeta del proyecto con la línea `GEMINI_API_KEY=tu_clave`.
3. **Solicitud interactiva al ejecutar la celda** (modo demostración → al hacer `Ctrl + Enter` aparecerá el cuadro para pegarla, sin que se vea en pantalla).

 **Aquí es donde debes pegar tu API Key cuando el sistema te lo pida al ejecutar la celda con `Ctrl + Enter`.**

In [ ]:
import os
from getpass import getpass

# Pedir la API Key directamente (no se muestra en pantalla)
print(" Pega tu API Key de Gemini y presiona Enter:")
gemini_key = getpass("GEMINI_API_KEY ➜ ")

# Guardarla como variable de entorno para que el SDK la use
os.environ["GEMINI_API_KEY"] = gemini_key

assert gemini_key.strip(), " La API Key está vacía. Reinicia la celda e ingrésala de nuevo."

print(f" Clave cargada correctamente — longitud: {len(gemini_key)} caracteres.")

🔑 Pega tu API Key de Gemini y presiona Enter:
✅ Clave cargada correctamente — longitud: 39 caracteres.


##  Paso 2 — Verificación del traductor

Vamos a usar `deep-translator` (que utiliza por debajo el motor web de Google Translate). Es **gratuito**, no requiere clave de API ni Docker, y funciona con una conexión a internet.

**Lo que conviene saber:**
-  No tiene un *rate-limit* estricto, pero por buenas prácticas dejaremos una pausa breve entre llamadas.
-  Soporta hasta **5 000 caracteres** por petición — más que suficiente para nuestros fragmentos.
-  Permite procesar lotes con `translate_batch()` si más adelante quieres acelerarlo.

Hagamos un *health check* rápido para confirmar que todo responde.

In [ ]:
from deep_translator import GoogleTranslator


def check_translator() -> bool:
    """Confirma que el servicio de traducción responde correctamente."""
    sample = "Knowledge is power."
    try:
        translation = GoogleTranslator(source="en", target="es").translate(sample)
        print("✅ Traductor operativo.")
        print(f"   Entrada : {sample}")
        print(f"   Salida  : {translation}")
        return True
    except Exception as err:
        print(f" El traductor no respondió: {err}")
        print("   Revisa tu conexión a internet y vuelve a intentar.")
        return False


check_translator()

✅ Traductor operativo.
   Entrada : Knowledge is power.
   Salida  : El conocimiento es poder.


True

##  Paso 3 — Lectura del documento PDF

Para esta demostración trabajaremos con un **documento corto** (artículo, manual o capítulo de pocas páginas). La librería `pypdf` extrae únicamente la **capa de texto** del PDF: ignora imágenes, fórmulas vectoriales e iconos, lo cual es perfecto para nosotros.

>  **Cambia la ruta `DOCUMENT_PATH`** para apuntar al archivo que mostrarás en clase. Recomendado: un PDF de 5 a 30 páginas para que el pipeline corra en minutos.

In [ ]:
from pypdf import PdfReader

#  Reemplaza esta ruta por la del documento que vas a usar en la demostración
DOCUMENT_PATH = r"../../../_data/Artificial_intelligence_applications_Latin.pdf"


def extract_pdf_content(file_path: str) -> str:
    """
    Recorre todas las páginas del PDF y devuelve el texto concatenado.
    Si una página no tiene capa de texto, la omite silenciosamente.
    """
    reader = PdfReader(file_path)
    page_count = len(reader.pages)
    print(f" Páginas detectadas: {page_count}")

    text_parts = []
    for page in reader.pages:
        page_text = page.extract_text() or ""
        text_parts.append(page_text)

    return "\n".join(text_parts)


# Ejecutamos la extracción
raw_text = extract_pdf_content(DOCUMENT_PATH)

print(f" Caracteres totales : {len(raw_text):,}")
print(f" Palabras aprox.    : {len(raw_text.split()):,}")
print("\n Vista previa (primeros 400 caracteres):")
print("-" * 60)
print(raw_text[:400])
print("-" * 60)

📄 Páginas detectadas: 20
 Caracteres totales : 67,725
 Palabras aprox.    : 9,886

🔍 Vista previa (primeros 400 caracteres):
------------------------------------------------------------
Artificial intelligence applications in Latin 
American higher education: a systematic review
Sdenka Zobeida Salas‑Pilco1*  and Yuqin Yang2* 
Introduction
Artificial intelligence (AI) has become increasingly important in recent decades. It is 
having a significant impact in numerous fields, such as medicine, finance, law, indus -
try, and entertainment (Amisha et al., 2019; Gade et al., 2020). Edu
------------------------------------------------------------


##  Paso 4 — Conteo de tokens

Los modelos de lenguaje no procesan palabras, sino **tokens**: pequeñas unidades que pueden ser una palabra, un trozo de palabra o un signo de puntuación. Necesitamos contarlos para asegurarnos de que cada fragmento (chunk) no supere el límite del modelo.

Usaremos el codificador `cl100k_base` (de OpenAI) como referencia, ya que se aproxima muy bien a la tokenización de Gemini y nos sirve para estimar tamaños.

In [ ]:
import tiktoken

# Cargamos un tokenizador estándar como medida de referencia
encoder = tiktoken.get_encoding("cl100k_base")


def count_tokens(text: str) -> int:
    """Devuelve la cantidad de tokens del texto según cl100k_base."""
    return len(encoder.encode(text))


total_tokens = count_tokens(raw_text)

print(f" Tokens del documento : {total_tokens:,}")
print(f" Límite por petición  : 8 192 tokens (gemini-embedding-001)")
print(f"  Por eso lo dividiremos en fragmentos en el siguiente paso.")

🔢 Tokens del documento : 17,292
📏 Límite por petición  : 8 192 tokens (gemini-embedding-001)
➡️  Por eso lo dividiremos en fragmentos en el siguiente paso.


##  Paso 5 — Segmentación en fragmentos (chunking)

Aquí cortamos el documento en piezas pequeñas usando `RecursiveCharacterTextSplitter`. Este *splitter* prueba **por orden** los siguientes separadores hasta encontrar uno que respete la longitud objetivo:

1. Doble salto de línea (`\n\n`) → entre párrafos.
2. Punto (`.`) → entre oraciones.
3. Salto de línea simple (`\n`).
4. Espacio en blanco.
5. Carácter por carácter (último recurso).

Así evitamos cortar palabras u oraciones por la mitad.

**Parámetros que vamos a usar:**

| Parámetro | Valor | Justificación |
|---|---|---|
| `chunk_size` | `200` tokens | Fragmentos cortos = búsqueda semántica más precisa |
| `chunk_overlap` | `25` tokens | Mantiene contexto compartido entre fragmentos vecinos |

>  Para un documento de demostración (5–30 páginas) esperamos entre **20 y 150 fragmentos**, lo cual es muy cómodo para el plan gratuito de Gemini.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=25,
    length_function=count_tokens,
    separators=["\n\n", ".", "\n", " ", ""],
)

# Metadatos comunes para todos los fragmentos
common_metadata = {
    "source": DOCUMENT_PATH,
    "language": "en->es",
    "pipeline": "rag-demo",
}

text_segments = splitter.create_documents([raw_text], metadatas=[common_metadata])

print(f"  Fragmentos generados : {len(text_segments)}")
print(f" Tamaño objetivo       : 200 tokens (con 25 de solapamiento)")

# Mostramos un fragmento de ejemplo
sample_idx = min(3, len(text_segments) - 1)
print(f"\n Ejemplo — fragmento #{sample_idx} (texto original):")
print("-" * 60)
print(text_segments[sample_idx].page_content)
print("-" * 60)

  Fragmentos generados : 107
 Tamaño objetivo       : 200 tokens (con 25 de solapamiento)

🔍 Ejemplo — fragmento #3 (texto original):
------------------------------------------------------------
. Open Access This article is licensed under a Creative Commons Attribution 4.0 International License, which permits 
use, sharing, adaptation, distribution and reproduction in any medium or format, as long as you give appropriate credit to the original 
author(s) and the source, provide a link to the Creative Commons licence, and indicate if changes were made. The images or other third 
party material in this article are included in the article’s Creative Commons licence, unless indicated otherwise in a credit line to the mate‑
rial. If material is not included in the article’s Creative Commons licence and your intended use is not permitted by statutory regulation or 
exceeds the permitted use, you will need to obtain permission directly from the copyright holder. To view a copy of this licenc

##  Paso 6 — Traducción de los fragmentos al español

Recorremos la lista de fragmentos y los traducimos uno por uno al español. Cada traducción se acompaña de:

-  Una **pausa breve** entre peticiones (para no parecer un bot).
-  Un **reintento con espera creciente** (*backoff*) si la red falla.
-  Una **caché en disco** (`segments_cache.pkl`): si el kernel se reinicia, no traducimos lo mismo dos veces.

>  Para un documento corto este paso toma pocos segundos. La caché solo aporta valor si decides repetir la demostración varias veces.

In [ ]:
import time
import pickle
import os
from tqdm.auto import tqdm
from deep_translator import GoogleTranslator

CACHE_FILE = "segments_cache.pkl"
DELAY_BETWEEN_CALLS = 0.25  # segundos


def translate_to_spanish(text: str, retries: int = 3) -> str:
    """Traduce un fragmento de inglés a español con reintentos automáticos."""
    for attempt in range(retries):
        try:
            translated = GoogleTranslator(source="en", target="es").translate(text)
            return translated or text
        except Exception as err:
            if attempt < retries - 1:
                time.sleep(2 * (attempt + 1))  # espera creciente
            else:
                print(f"  Traducción fallida tras {retries} intentos: {str(err)[:80]}")
    return text  # fallback: devolvemos el original


def persist_cache(payload, path: str) -> None:
    """Guarda el progreso en disco para no repetir trabajo."""
    with open(path, "wb") as fh:
        pickle.dump(payload, fh)


# 1) Si existe caché, la cargamos y completamos lo que falte
if os.path.exists(CACHE_FILE):
    with open(CACHE_FILE, "rb") as fh:
        translated_segments = pickle.load(fh)
    print(f"  Caché encontrada con {len(translated_segments)} fragmentos.")

    pending = text_segments[len(translated_segments):]
    if pending:
        print(f"   Continuando con los {len(pending)} fragmentos restantes...")
        for piece in tqdm(pending, desc="🌍 Traduciendo (continuación)"):
            translated_segments.append({
                "text": translate_to_spanish(piece.page_content),
                "original": piece.page_content,
                "metadata": piece.metadata,
            })
            time.sleep(DELAY_BETWEEN_CALLS)
        persist_cache(translated_segments, CACHE_FILE)

# 2) Si no hay caché, traducimos desde cero
else:
    translated_segments = []
    for idx, piece in enumerate(tqdm(text_segments, desc=" Traduciendo EN→ES")):
        translated_segments.append({
            "text": translate_to_spanish(piece.page_content),
            "original": piece.page_content,
            "metadata": piece.metadata,
        })
        time.sleep(DELAY_BETWEEN_CALLS)
        # Guardamos cada 25 fragmentos para no perder el progreso
        if (idx + 1) % 25 == 0:
            persist_cache(translated_segments, CACHE_FILE)
    persist_cache(translated_segments, CACHE_FILE)
    print(f" {len(translated_segments)} traducciones guardadas en '{CACHE_FILE}'.")

# Vista previa de un fragmento traducido
preview_idx = min(3, len(translated_segments) - 1)
print(f"\n Fragmento #{preview_idx} traducido al español:")
print("-" * 60)
print(translated_segments[preview_idx]["text"])
print("-" * 60)

♻️  Caché encontrada con 107 fragmentos.

🔍 Fragmento #3 traducido al español:
------------------------------------------------------------
. Acceso abierto Este artículo tiene una licencia internacional Creative Commons Attribution 4.0, que permite 
usar, compartir, adaptar, distribuir y reproducir en cualquier medio o formato, siempre y cuando se dé el crédito apropiado al original 
autor(es) y la fuente, proporcione un enlace a la licencia Creative Commons e indique si se realizaron cambios. Las imágenes u otros terceros 
El material de las partes en este artículo está incluido en la licencia Creative Commons del artículo, a menos que se indique lo contrario en una línea de crédito al material.
rial. Si el material no está incluido en la licencia Creative Commons del artículo y su uso previsto no está permitido por la regulación legal o 
excede el uso permitido, necesitará obtener permiso directamente del titular de los derechos de autor. Para ver una copia de esta licencia, visite 

##  Paso 7 — Generación de embeddings con Gemini

Un **embedding** es un vector numérico de varias dimensiones que representa el significado semántico de un texto. La idea clave es:

> Dos textos con sentido parecido tienen vectores cercanos en el espacio, aunque usen palabras distintas.

Trabajaremos con el modelo **`gemini-embedding-001`** y elegiremos **768 dimensiones** (un buen equilibrio entre calidad y costo de almacenamiento).

**Detalle importante — `task_type`:**

| Tipo | Cuándo se usa |
|---|---|
| `RETRIEVAL_DOCUMENT` | Al **indexar** los fragmentos de la base de conocimiento |
| `RETRIEVAL_QUERY` | Al **consultar** con la pregunta del usuario |

Esta distinción mejora la precisión: el modelo sabe si está optimizando para *almacenar* o para *buscar*.

Como ChromaDB no incluye una `EmbeddingFunction` lista para el SDK nuevo de Gemini, **definimos una propia** compatible con su interfaz, con throttling automático para respetar el límite gratuito.

In [ ]:
import time
from google import genai
from google.genai import types
from chromadb import EmbeddingFunction, Documents, Embeddings


# Cliente único de Gemini, reutilizable durante toda la sesión
gemini = genai.Client(api_key=gemini_key)

# Configuración de modelos
EMBEDDING_MODEL = "gemini-embedding-001"
EMBEDDING_DIMENSIONS = 768          # opciones soportadas: 768, 1536, 3072
GENERATION_MODEL = "gemini-2.5-flash"

# Pausa entre embeddings → respeta el límite del free tier (60 req/min)
EMBEDDING_PAUSE = 1.1


class GeminiEmbedder(EmbeddingFunction):
    """
    Función de embeddings compatible con ChromaDB que utiliza el SDK
    google-genai. Aplica throttling y reintentos automáticos.
    """

    def __init__(self, task_type: str = "RETRIEVAL_DOCUMENT"):
        self.task_type = task_type

    def __call__(self, input: Documents) -> Embeddings:
        vectors = []
        for text in input:
            for attempt in range(3):
                try:
                    response = gemini.models.embed_content(
                        model=EMBEDDING_MODEL,
                        contents=text,
                        config=types.EmbedContentConfig(
                            task_type=self.task_type,
                            output_dimensionality=EMBEDDING_DIMENSIONS,
                        ),
                    )
                    vectors.append(response.embeddings[0].values)
                    time.sleep(EMBEDDING_PAUSE)
                    break
                except Exception as err:
                    print(f"⚠️  Reintento {attempt + 1}/3 → {str(err)[:80]}")
                    time.sleep(5)
            else:
                raise RuntimeError("❌ El servicio de embeddings falló tras 3 intentos.")
        return vectors

    def name(self) -> str:
        return EMBEDDING_MODEL


# Dos instancias: una para indexar y otra para consultar
embedder_documents = GeminiEmbedder(task_type="RETRIEVAL_DOCUMENT")
embedder_queries   = GeminiEmbedder(task_type="RETRIEVAL_QUERY")

print(" Embeddings listos.")
print(f"   • Modelo       : {EMBEDDING_MODEL}")
print(f"   • Dimensiones  : {EMBEDDING_DIMENSIONS}")
print(f"   • Throttling   : 1 petición cada {EMBEDDING_PAUSE}s (~{int(60/EMBEDDING_PAUSE)} por minuto)")

✅ Embeddings listos.
   • Modelo       : gemini-embedding-001
   • Dimensiones  : 768
   • Throttling   : 1 petición cada 1.1s (~54 por minuto)


##  Paso 8 — Base vectorial persistente con ChromaDB

ChromaDB nos permite **guardar los vectores en disco**. Si reinicias el kernel del notebook, no tendrás que volver a calcular embeddings: la colección queda lista para usarse.

-  Carpeta de persistencia: `./chroma_storage`
-  Métrica de similitud: `cosine` (la indicada para embeddings normalizados)
-  Nombre de colección: `rag_demo_collection`

In [ ]:
import chromadb

CHROMA_PATH = "./chroma_storage"
COLLECTION = "rag_demo_collection"

# Cliente persistente: los datos sobreviven a reinicios del kernel
chroma = chromadb.PersistentClient(path=CHROMA_PATH)

# get_or_create_collection: si ya existe, la reutiliza sin re-indexar
vector_store = chroma.get_or_create_collection(
    name=COLLECTION,
    embedding_function=embedder_documents,
    metadata={"hnsw:space": "cosine"},
)

print(f" Colección activa : '{COLLECTION}'")
print(f" Persistencia en  : {CHROMA_PATH}")
print(f" Documentos hoy   : {vector_store.count()}")

📦 Colección activa : 'rag_demo_collection'
📍 Persistencia en  : ./chroma_storage
🔢 Documentos hoy   : 107


##  Paso 9 — Indexar los fragmentos en ChromaDB

Si la colección ya tiene fragmentos indexados (porque corriste esta celda anteriormente), **el bloque saltará** el trabajo redundante. Si está vacía, indexa todos los fragmentos llamando al embebedor de Gemini con throttling.

>  **Tiempo estimado para la demostración** (documento corto, ~30–100 fragmentos): aproximadamente 1 a 2 minutos.

In [ ]:
from tqdm.auto import tqdm

BATCH_SIZE = 10  # tamaño del lote para indexar


def already_indexed(store) -> int:
    """Devuelve cuántos documentos ya hay en la colección."""
    return store.count()


existing_count = already_indexed(vector_store)

if existing_count >= len(translated_segments):
    print(f"♻️  La colección ya contiene {existing_count} fragmentos. No se reindexa.")
else:
    pending_segments = translated_segments[existing_count:]
    minutes_est = (len(pending_segments) * EMBEDDING_PAUSE) / 60

    print(f"📥 Se indexarán {len(pending_segments)} fragmentos nuevos.")
    print(f"   Tiempo estimado: ~{minutes_est:.1f} minutos.\n")

    for batch_start in tqdm(range(0, len(pending_segments), BATCH_SIZE),
                             desc="📥 Indexando"):
        batch = pending_segments[batch_start: batch_start + BATCH_SIZE]
        offset = existing_count + batch_start

        vector_store.add(
            documents=[item["text"] for item in batch],
            metadatas=[item["metadata"] for item in batch],
            ids=[f"segment_{offset + j}" for j in range(len(batch))],
        )

    print(f"\n Indexado completo. Total en colección: {vector_store.count()}")

♻️  La colección ya contiene 107 fragmentos. No se reindexa.


##  Paso 10 — Búsqueda semántica + generación de respuesta

Aquí ensamblamos las dos piezas centrales del sistema RAG:

1. **`retrieve_relevant(question, k)`**  
   Convierte la pregunta en embedding (con `task_type=RETRIEVAL_QUERY`) y devuelve los `k` fragmentos más cercanos.

2. **`generate_answer(question, k)`**  
   Toma esos fragmentos como **contexto**, los entrega a `gemini-2.5-flash` y obtiene una respuesta en español que **se basa exclusivamente en el documento**, lo cual reduce drásticamente las alucinaciones del modelo.

In [ ]:
def retrieve_relevant(question: str, k: int = 4) -> list[dict]:
    """Devuelve los k fragmentos más relevantes para una pregunta."""
    query_vector = embedder_queries([question])[0]

    results = vector_store.query(
        query_embeddings=[query_vector],
        n_results=k,
    )

    return [
        {"text": doc, "metadata": meta, "distance": dist}
        for doc, meta, dist in zip(
            results["documents"][0],
            results["metadatas"][0],
            results["distances"][0],
        )
    ]


# Instrucciones que guían el comportamiento del LLM
INSTRUCTIONS = """Eres un asistente experto que responde preguntas
basándote EXCLUSIVAMENTE en los fragmentos del documento que se te entregan.

Reglas estrictas:
- Responde siempre en español, con un tono claro y didáctico.
- Si la información no aparece en los fragmentos, responde:
  "No encontré esa información en el documento."
- No inventes datos ni completes con conocimiento general.
- Cuando ayude a la comprensión, indica brevemente de qué fragmento proviene la idea.
"""


def generate_answer(question: str, k: int = 4) -> dict:
    """Pipeline RAG: recupera contexto y genera la respuesta final."""
    retrieved = retrieve_relevant(question, k=k)

    context_block = "\n\n---\n\n".join(
        f"[Fragmento {i + 1}] {item['text']}"
        for i, item in enumerate(retrieved)
    )

    full_prompt = f"""{INSTRUCTIONS}

CONTEXTO DEL DOCUMENTO:
{context_block}

PREGUNTA:
{question}

RESPUESTA:"""

    completion = gemini.models.generate_content(
        model=GENERATION_MODEL,
        contents=full_prompt,
    )

    return {"answer": completion.text, "sources": retrieved}


# Mini-prueba sin interfaz gráfica
print(" Prueba rápida del pipeline RAG\n")
demo = generate_answer("¿De qué trata este documento?", k=3)
print(" RESPUESTA:")
print(demo["answer"])
print("\n FUENTES (top 3):")
for n, src in enumerate(demo["sources"], 1):
    print(f"   {n}. distancia={src['distance']:.4f} → {src['text'][:120]}...")

🧪 Prueba rápida del pipeline RAG

📝 RESPUESTA:
No encontré esa información en el documento. Los fragmentos proporcionados describen la licencia de acceso abierto del artículo (Fragmento 1) y presentan una lista de referencias bibliográficas a otras publicaciones y fuentes (Fragmento 2 y Fragmento 3), pero no el tema principal de este documento en particular.

📚 FUENTES (top 3):
   1. distancia=0.3188 → . Acceso abierto Este artículo tiene una licencia internacional Creative Commons Attribution 4.0, que permite 
usar, com...
   2. distancia=0.3250 → . (2017). Detección y análisis de comunidades científicas en la bibliografía. 
base de datos: SCOPUS. En Actas de la cua...
   3. distancia=0.3277 → . (2017). Representación de América Latina 
Programas universitarios en una red semántica. En Proceedings 2017 conferenc...


##  Paso 11 — Interfaz interactiva con `ipywidgets`

Para que la demostración sea atractiva en clase, montamos una **mini-aplicación** dentro del propio notebook: un cuadro de texto, un selector de cuántos fragmentos recuperar, y un panel de respuesta con buen formato.

>  Si los widgets no aparecen en VS Code, abre la paleta de comandos (`Ctrl + Shift + P`) y ejecuta `Jupyter: Restart Kernel`. Luego vuelve a correr esta celda.

In [1]:
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

# === Encabezado visual ===
header = widgets.HTML("""
<div style='background: linear-gradient(135deg, #4f46e5 0%, #9333ea 100%);
            padding: 22px; border-radius: 14px; color: white;
            font-family: -apple-system, "Segoe UI", sans-serif; margin-bottom: 14px;
            box-shadow: 0 4px 12px rgba(79, 70, 229, 0.25);'>
    <h2 style='margin:0; font-weight:600;'>🎓 Asistente RAG sobre tu documento</h2>
    <p style='margin:6px 0 0 0; opacity:0.92; font-size:0.95em;'>
        Demostración educativa · Embeddings Gemini + ChromaDB
    </p>
</div>
""")

# === Controles ===
input_question = widgets.Textarea(
    placeholder="Escribe aquí tu pregunta... ejemplo: ¿Cuáles son las ideas principales?",
    layout=widgets.Layout(width="100%", height="90px"),
    description="❓",
)

slider_k = widgets.IntSlider(
    value=4, min=1, max=8, step=1,
    description="Fragmentos a recuperar:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="55%"),
)

btn_ask = widgets.Button(
    description=" Preguntar",
    button_style="primary",
    layout=widgets.Layout(width="180px", height="42px"),
)

btn_clear = widgets.Button(
    description="🧹 Limpiar",
    button_style="",
    layout=widgets.Layout(width="120px", height="42px"),
)

response_panel = widgets.Output()


def render_response(question: str, payload: dict) -> None:
    """Construye y muestra el HTML con la respuesta y las fuentes."""
    sources_html = ""
    for i, src in enumerate(payload["sources"], 1):
        sources_html += f"""
        <details style='margin: 8px 0; padding: 10px; background: #f3f4f6;
                        border-left: 3px solid #4f46e5; border-radius: 6px;'>
            <summary style='cursor: pointer; font-weight: 600; color: #4b5563;'>
                 Fragmento {i} — distancia: {src['distance']:.4f}
            </summary>
            <p style='margin-top: 8px; color: #1f2937; font-size: 0.92em; line-height: 1.55;'>
                {src['text']}
            </p>
        </details>
        """

    html_output = f"""
    <div style='font-family: -apple-system, "Segoe UI", sans-serif; max-width: 920px;'>
        <div style='background:#fff;padding:18px;border-radius:10px;
                    border:1px solid #e5e7eb;margin-bottom:14px;
                    box-shadow:0 2px 8px rgba(0,0,0,0.04);'>
            <div style='color:#4f46e5;font-weight:600;font-size:0.82em;
                        text-transform:uppercase;letter-spacing:0.5px;'>
                Pregunta
            </div>
            <div style='font-size:1.08em;color:#111827;margin-top:6px;'>{question}</div>
        </div>

        <div style='background:#fff;padding:20px;border-radius:10px;
                    border:1px solid #e5e7eb;margin-bottom:14px;
                    box-shadow:0 2px 8px rgba(0,0,0,0.04);'>
            <div style='color:#10b981;font-weight:600;font-size:0.82em;
                        text-transform:uppercase;letter-spacing:0.5px;'>
                 Respuesta del asistente
            </div>
            <div style='font-size:1.04em;color:#111827;margin-top:10px;
                        line-height:1.7;white-space:pre-wrap;'>{payload["answer"]}</div>
        </div>

        <div style='background:#fff;padding:16px;border-radius:10px;
                    border:1px solid #e5e7eb;'>
            <div style='color:#6b7280;font-weight:600;font-size:0.82em;
                        text-transform:uppercase;letter-spacing:0.5px;
                        margin-bottom:8px;'>
                 Fuentes recuperadas
            </div>
            {sources_html}
        </div>
    </div>
    """
    display(HTML(html_output))


def handle_ask(_):
    text = input_question.value.strip()
    if not text:
        with response_panel:
            clear_output()
            print("⚠️  Por favor escribe una pregunta primero.")
        return

    with response_panel:
        clear_output()
        print("⏳ Buscando fragmentos relevantes y generando la respuesta...")
        result = generate_answer(text, k=slider_k.value)
        clear_output()
        render_response(text, result)


def handle_clear(_):
    input_question.value = ""
    with response_panel:
        clear_output()


btn_ask.on_click(handle_ask)
btn_clear.on_click(handle_clear)

# Disposición final de la UI
ui_layout = widgets.VBox([
    header,
    input_question,
    widgets.HBox([slider_k]),
    widgets.HBox([btn_ask, btn_clear]),
    response_panel,
])

display(ui_layout)

##  Resumen y cierre

 construir un **sistema RAG completo**,
| Etapa del pipeline | Tecnología utilizada |
|---|---|
| Lectura del PDF | `pypdf` |
| Conteo de tokens | `tiktoken` |
| Segmentación | `langchain RecursiveCharacterTextSplitter` |
| Traducción | `deep-translator` (Google web) |
| Embeddings | `gemini-embedding-001` (768 dim) |
| Base vectorial | `ChromaDB` persistente |
| LLM generador | `gemini-2.5-flash` |
| Interfaz | `ipywidgets` |

---
